# Projet de comparaison de classification de photos de chats et chiens 

## Preparation des données

### Import de paquets

In [ ]:
import os
import random
import pandas as pd
import numpy as np
import torch
from torch import randperm
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, SubsetRandomSampler
from torchvision import datasets, transforms, models
from torchvision.transforms import v2
from torchmetrics.classification import BinaryAccuracy, BinaryConfusionMatrix
from torch.optim.lr_scheduler import StepLR
from sklearn.metrics import precision_score, recall_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import my_helper
import logging
from tqdm import tqdm
import shutil
from glob import glob
import optuna
import optuna.trial
import mlflow
import mlflow.pytorch
from packaging import requirements
from pprint import pformat
from IPython.display import display, Image

### Ajout de seed pour la reproductibilité

In [ ]:
def set_seed(seed=42):
    
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(42)

### vérification et choix du GPU si disponible

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Utilisation du périphérique : {device}")

### Réaménagement des données 

Ce réaménagement va permettre de mieux structurer les données dans les dossiers et permettra de mieux splitter les données en train, validation et test

In [ ]:
source_dir = os.getcwd()+"\\data\\Cat_Dog_data"
target_dir = os.getcwd()+ "\\data\\Cat_Dog_ordered"

os.makedirs(target_dir, exist_ok=True)

for label in ["cat", "dog"]:
    os.makedirs(os.path.join(target_dir, label), exist_ok=True)
print("Début du déplacement des images...")
search_pattern = os.path.join(source_dir, "*", "*", "*")
files = glob(search_pattern)

moved_count = 0

for file_path in files:
    if os.path.isfile(file_path):
        path_parts = file_path.split(os.sep)
        label = path_parts[-2] 
        filename = path_parts[-1]
        if label in ["cat", "dog"]:
            subfolder = path_parts[-3]
            new_filename = f"{subfolder}_{filename}"      
            destination = os.path.join(target_dir, label, new_filename)          
            shutil.move(file_path, destination)
            moved_count += 1

print(f"Terminé ! {moved_count} images ont été déplacées avec succès dans '{target_dir}'.")

### création de la liste des paramètres

In [ ]:
params = {
    "Experiment": "CNN_Cats&Dogs_Experiments",
    "data_dir": "data/Cat_Dog_ordered",
    "batch_size": 32,
    "num_workers": 4,
    "train_size": 0.8,
    "val_size": 0.1,
    "test_size": 0.1,
    "epochs": 20,
    "dropout": 0.5,
    "save_path_scratch": "best_scratch_model.pth",
    "save_path_transfert": "best_transfer_model.pth"
}

### Fonctions de transformation et augmentation des données

Préparation des fonctions de transformation et augmentation de données avec redimensionnement, 50% aléatoire de rotation de 15%, rotation horizontale de l'image, ainsi que normalisation de l'image pour train et redimensionnement et normalisation des données de test et validation.

In [ ]:
train_transforms = v2.Compose([
    v2.Resize((224, 224)),
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomRotation(degrees=15),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],std=[0.229, 0.224, 0.225])
])
test_val_transforms = v2.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])




### creation des dataloaders et visualisation de quelques images

In [ ]:

def get_data_loaders(data_dir, batch_size, train_size, valid_size, num_workers, train_transforms, test_val_transforms):
    train_dataset = datasets.ImageFolder(root=data_dir, transform=train_transforms)
    test_val_dataset = datasets.ImageFolder(root=data_dir, transform=test_val_transforms)
    
    n_tot = len(train_dataset)
    
    # 2. Calculer la taille des segments (80% / 10% / 10%)
    split_train = int(np.floor(train_size * n_tot))
    split_val = int(np.floor((train_size + valid_size) * n_tot))
    
    shuffled_indices = randperm(n_tot).tolist()
    
    train_idx = shuffled_indices[:split_train]
    valid_idx = shuffled_indices[split_train:split_val]
    test_idx = shuffled_indices[split_val:]
    

    train_sampler = SubsetRandomSampler(train_idx)
    valid_sampler = SubsetRandomSampler(valid_idx)
    test_sampler = SubsetRandomSampler(test_idx)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=train_sampler, num_workers=num_workers, persistent_workers=True, pin_memory=True,prefetch_factor=2)
    valid_loader = DataLoader(test_val_dataset, batch_size=batch_size, sampler=valid_sampler, num_workers=num_workers, persistent_workers=True, pin_memory=True,prefetch_factor=2)
    test_loader = DataLoader(test_val_dataset, batch_size=batch_size, sampler=test_sampler, num_workers=num_workers, persistent_workers=True, pin_memory=True,prefetch_factor=2)
    
    return train_loader, valid_loader, test_loader

train_dl, valid_dl, test_dl = get_data_loaders(
    data_dir=params["data_dir"],
    batch_size=params["batch_size"],
    train_size=params["train_size"],
    valid_size=params["val_size"],
    num_workers=params["num_workers"],
    train_transforms=train_transforms,
    test_val_transforms=test_val_transforms
)

data_loaders = {
    'train': train_dl,
    'val': valid_dl,
    'test': test_dl
}

In [ ]:
classes = train_dl.dataset.classes
print(f"Classes : {classes}")
num_classes = len(classes)
print(f"Nombre de classes : {num_classes}")

Nombre de batchs pour train, val et test

In [ ]:
print(len(data_loaders['train']), len(data_loaders['val']), len(data_loaders['test']))

Test des loaders

In [ ]:
images, labels = next(iter(data_loaders["train"]))
fig, subs = plt.subplots(2, 10, figsize=(25, 4))
print(labels)
for i, sub in enumerate(subs.flatten()):
    my_helper.imshow(images[i], sub)
    sub.set_title([classes[labels[i]]])

In [ ]:
taille_batch = data_loaders["train"].batch_size

In [ ]:
images.size()

In [ ]:
batch = next(iter(data_loaders["train"]))

if isinstance(batch, torch.Tensor):
    taille_octets = batch.element_size() * batch.nelement()
else:

    taille_octets = sum(b.element_size() * b.nelement() for b in batch if isinstance(b, torch.Tensor))

# Conversion en KB
taille_kb = taille_octets / 1024
taille_mb = taille_kb / 1024

print(f"Taille du batch : {taille_mb:.2f} MB")

In [ ]:
image, label = next(iter(data_loaders["val"]))
my_helper.imshow(image[0],title=f"Label: {label[0]}")

In [ ]:
image, label = next(iter(data_loaders["test"]))
my_helper.imshow(image[0],title=f"Label: {label[0]}")

 ### Conception du modèle

In [ ]:
class CNNScratch(nn.Module):
    def __init__(self):
        super(CNNScratch, self).__init__()
        
        self.features = nn.Sequential(
            # Bloc 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            # Bloc 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            # Bloc 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 512),
            nn.ReLU(),
            nn.Dropout(0.5), 
            nn.Linear(512, 2) 
        )

    def forward(self, x):
        return self.classifier(self.features(x))

### Conception des alogrithmes d'entraînement et d'évaluation ainsi que des fonctions de test et d'affichage de la matrice de confusion

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, acc_metric):
    model.train()
    acc_metric.reset()
    running_loss = 0.0
    for imgs, targets in tqdm(loader, desc="Train", ncols=100):
        imgs, targets = imgs.to(device,non_blocking=True), targets.to(device,non_blocking=True)
        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        preds = torch.argmax(logits, dim=1)
        acc_metric.update(preds, targets)
    return running_loss / len(loader.sampler), acc_metric.compute().item()

def evaluate(model, loader, criterion, acc_metric):
    model.eval()
    acc_metric.reset()
    running_loss = 0.0
    with torch.no_grad():
        for imgs, targets in tqdm(loader, desc="Valid", ncols=100):
            imgs, targets = imgs.to(device,non_blocking=True), targets.to(device,non_blocking=True)
            logits = model(imgs)
            loss = criterion(logits, targets)
            running_loss += loss.item() * imgs.size(0)
            preds = torch.argmax(logits, dim=1)
            acc_metric.update(preds, targets)
    return running_loss / len(loader.sampler), acc_metric.compute().item()

def test_and_confusion(model, loader):
    model.eval()
    cm_metric = BinaryConfusionMatrix().to(device)
    test_acc.reset()
    with torch.no_grad():
        for imgs, targets in tqdm(loader, desc="Test", ncols=100):
            imgs, targets = imgs.to(device), targets.to(device)
            logits = model(imgs)
            preds = torch.argmax(logits, dim=1)
            cm_metric.update(preds, targets)
            test_acc.update(preds, targets)
    cm = cm_metric.compute().cpu().numpy()
    acc = test_acc.compute().item()
    targets_cpu = targets.cpu().numpy()
    preds_cpu = preds.cpu().numpy()
    precision = precision_score(targets_cpu, preds_cpu, average='binary')
    recall = recall_score(targets_cpu, preds_cpu, average='binary')
    print(f"\nTest Accuracy : {acc:.4f}")
    print(f"Test Precision : {precision:.4f}")
    print(f"Test Recall : {recall:.4f}")
    print(f"Confusion Matrix :\n{cm}")
    return cm , f"{acc:.4f}", f"{precision:.4f}", f"{recall:.4f}"

def plot_confusion_matrix(cm, classes):
    confusion_matrix = pd.DataFrame(cm, index=classes, columns=classes)
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(confusion_matrix, annot=True, fmt='d', cmap="Blues", ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    fig.savefig('confusion_matrix.png')
    plt.close()

def accuracy_by_class(cm, classes):
    print("\nAccuracy by class:\n")
    for i, class_name in enumerate(classes):
        correct_predictions = cm[i][i]
        total_predictions = cm[i].sum()
        accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0
        print(f"    {class_name:11s}: {accuracy:.2f}")

def visualize_predictions(model, loader, classes, n_batches=1):
    model.eval()
    dataiter = iter(loader)
    for _ in range(n_batches):
        images, labels = next(dataiter)
        images, labels = images.to(device), labels.to(device)
        with torch.no_grad():
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
        fig, subs = plt.subplots(2, 10, figsize=(25, 4))
        for i, ax in enumerate(subs.flatten()):
            img = images[i].cpu().numpy()
            img = img / 2 + 0.5  # unnormalize
            ax.imshow(np.transpose(img, (1, 2, 0)))
            ax.set_title(f"{classes[preds[i]]} ({classes[labels[i]]})",
                         color=("green" if preds[i] == labels[i] else "red"))
            ax.axis("off")
        plt.savefig('prediction_visualized.png')
        plt.close()

In [ ]:
criterion = nn.CrossEntropyLoss()
train_acc = BinaryAccuracy().to(device)
valid_acc = BinaryAccuracy().to(device)
test_acc = BinaryAccuracy().to(device)

### conception des fonctions d'essai et de recherche d'hyperparamètres

In [ ]:
def suggest_hyperparameters(trial):
    lr = trial.suggest_categorical("lr", [1e-4, 1e-3, 1e-2])

    optimizer_name = trial.suggest_categorical("optimizer_name", ["Adam", "SGD"])

    print(f"Hyperparamètres à tester: \n{pformat(trial.params)}")
    return lr, optimizer_name

In [ ]:
def objective(trial):
    print("\n********************************\n")
    best_val = float("inf")
    lr, optimizer_name = suggest_hyperparameters(trial)
    print(f"Training on {device}")
    modele = CNNScratch().to(device)
    scripted = torch.jit.script(modele)
    torch.jit.save(scripted, "fromscratch_network.pt")
    if optimizer_name == "Adam":
        optimizer = optim.Adam(modele.parameters(), lr=lr)
    if optimizer_name == "SGD":
        optimizer = optim.SGD(modele.parameters(), lr=lr,momentum=0.9)
    scheduler = StepLR(optimizer, step_size=1, gamma=0.7)
    optuna_epochs = 10
    for epoch in range(1, optuna_epochs + 1):
        tr_loss, tr_acc = train_one_epoch(modele, data_loaders["train"], criterion, optimizer, train_acc)
        val_loss, val_acc = evaluate(modele, data_loaders["val"], criterion, valid_acc)
        print(f"Epoch {epoch:02d} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | val loss {val_loss:.4f} acc {val_acc:.4f}")
        if val_loss < best_val:
            best_val = val_loss
        scheduler.step()
        trial.report(val_loss, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
    return best_val
study = optuna.create_study(study_name="optuna-cnn-fromscratch", direction="minimize")
study.optimize(objective, n_trials=6)

    # Print optuna study statistics
print("\n++++++++++++++++++++++++++++++++++\n")
print("Study statistics: ")
print("  Number of finished trials: ", len(study.trials))

print("Best trial:")
trial = study.best_trial

print("  Trial number: ", trial.number)
print("  Loss (trial value): ", trial.value)

print("  Params: ")
for key, value in trial.params.items():
    print("    {}: {}".format(key, value))

    

### récupérations des meilleurs hyperparamètres et entraînement du modèle final from scratch avec journalisation sur MLFlow en local

In [ ]:
best_params_scratch = {
    "lr": trial.params["lr"],
    "optimizer_name": trial.params["optimizer_name"]
}

In [ ]:
val_loss_scratch = []
val_acc_scratch = []
test_acc_scratch = []
test_precision_scratch = []
test_recall_scratch = []

In [ ]:
def fromscratch():
    mlflow.set_tracking_uri("http://localhost:5000")
    mlflow.set_experiment(params["Experiment"])
    mlflow.config.enable_system_metrics_logging()
    print("✓ Successfully connected to MLflow!")
    with open('requirements.txt', 'r') as file:
        requirements_info = [line.strip() for line in file]
    
    best_val = float("inf")
    with mlflow.start_run(run_name="CNN_Scratch_Entrainement_Final"):
        mlflow.log_param("device", device)
        mlflow.log_params(params)
        lr = best_params_scratch["lr"]
        optimizer_name = best_params_scratch["optimizer_name"]
        mlflow.log_params(best_params_scratch)
        print(f"Hyperparamètres utilisés pour l'entraînement final: \n{pformat(best_params_scratch)}")
        print(f"Training on {device}")
        modele = CNNScratch().to(device)
        if optimizer_name == "Adam":
            optimizer = optim.Adam(modele.parameters(), lr=lr)
        if optimizer_name == "SGD":
            optimizer = optim.SGD(modele.parameters(), lr=lr,momentum=0.9)
        scheduler = StepLR(optimizer, step_size=1, gamma=0.7)
        for epoch in range(1, params["epochs"] + 1):
            tr_loss, tr_acc = train_one_epoch(modele, data_loaders["train"], criterion, optimizer, train_acc)
            val_loss, val_acc = evaluate(modele, data_loaders["val"], criterion, valid_acc)
            print(f"Epoch {epoch:02d} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | val loss {val_loss:.4f} acc {val_acc:.4f}")
            mlflow.log_metrics({"train_loss": tr_loss, "train_accuracy": tr_acc, "val_loss": val_loss, "val_accuracy": val_acc}, step=epoch)
            val_loss_scratch.append(val_loss)
            val_acc_scratch.append(val_acc)
            if val_loss < best_val:
                best_val = val_loss
                torch.save(modele.state_dict(), params["save_path_scratch"])
                print("New best model saved!")
            scheduler.step()
        torch.load("best_scratch_model.pth")
        mlflow.pytorch.log_model(modele, name="final_model", pip_requirements="requirements.txt")
    
        # Final test + confusion matrix
        print("\nEvaluating on test set …")
        cm, test_acc,test_precision, test_recall = test_and_confusion(modele, data_loaders["test"])

        test_acc_scratch.append(test_acc)
        test_precision_scratch.append(test_precision)
        test_recall_scratch.append(test_recall)

        mlflow.log_metric("test_accuracy", test_acc)
        mlflow.log_metric("test_precision", test_precision)
        mlflow.log_metric("test_recall", test_recall)
        plot_confusion_matrix(cm, classes)
        os.replace('confusion_matrix.png', 'confusion_matrix_scratch.png')
        mlflow.log_artifact(os.getcwd()+"\\confusion_matrix_scratch.png", artifact_path="confusion_matrix")
        accuracy_by_class(cm,classes)

        visualize_predictions(modele,data_loaders['test'],classes)
        os.replace('prediction_visualized.png', 'prediction_visualized_scratch.png')
        mlflow.log_artifact(os.getcwd()+"\\prediction_visualized_scratch.png", artifact_path="prediction_visualized")

In [ ]:
fromscratch()


In [ ]:
display(Image(filename='confusion_matrix_scratch.png', width=600))

In [ ]:
display(Image(filename='prediction_visualized_scratch.png', height=600))

### conception des paramètres d'éssai pour la recherche des meilleurs hyperparamètres pour le tansfert learning


In [ ]:
criterion = nn.CrossEntropyLoss()
train_acc = BinaryAccuracy().to(device)
valid_acc = BinaryAccuracy().to(device)
test_acc = BinaryAccuracy().to(device)

In [ ]:
def suggestion_hyperparametres(trial):
    lr = trial.suggest_categorical("lr", [1e-4, 1e-3, 1e-2])
    optimizer_name = trial.suggest_categorical("optimizer_name", ["Adam", "SGD"])
    couches_gel = trial.suggest_int("couches_gel", 1,8)
    print(f"Hyperparamètres à tester: \n{pformat(trial.params)}")
    return lr, optimizer_name, couches_gel
    

In [ ]:
def objective(trial):
    
    print("\n********************************\n")
    best_val = float("inf")
    lr, optimizer_name, couches_gel = suggestion_hyperparametres(trial)
    print(f"Training on {device}")
    modele = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT).to(device)
    features = list(modele.features.children())

    total_features = len(features)

    layers_to_freeze = min(couches_gel, total_features)

    for i in range(layers_to_freeze):
        for param in features[i].parameters():
            param.requires_grad = False
    
    for i in range(layers_to_freeze, total_features):
        for param in features[i].parameters():
            param.requires_grad = True

    if optimizer_name == "Adam":
        optimizer = optim.Adam(modele.parameters(), lr=lr)
    if optimizer_name == "SGD":
        optimizer = optim.SGD(modele.parameters(), lr=lr,momentum=0.9)
    
    scheduler = StepLR(optimizer, step_size=1, gamma=0.7)
    nbre_param = modele.classifier[3].in_features
    modele.classifier[3] = nn.Linear(nbre_param, num_classes)
    modele = modele.to(device)
    
    optuna_epochs = 10
    for epoch in range(1, optuna_epochs + 1):
        tr_loss, tr_acc = train_one_epoch(modele, data_loaders["train"], criterion, optimizer, train_acc)
        val_loss, val_acc = evaluate(modele, data_loaders["val"], criterion, valid_acc)
        print(f"Epoch {epoch:02d} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | val loss {val_loss:.4f} acc {val_acc:.4f}")
        if val_loss < best_val:
            best_val = val_loss
        scheduler.step()
        trial.report(val_loss, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
    return best_val

In [ ]:
study = optuna.create_study(study_name="optuna-cnn-transfer", direction="minimize")
study.optimize(objective, n_trials=6)

In [ ]:
print("\n++++++++++++++++++++++++++++++++++\n")
print("Study statistics: ")
print("  Number of finished trials: ", len(study.trials))

print("Best trial:")
trial = study.best_trial

print("  Trial number: ", trial.number)
print("  Loss (trial value): ", trial.value)

print("  Params: ")
for key, value in trial.params.items():
    print("    {}: {}".format(key, value))


### Récupération des meilleurs hyperparamètres et entrainement du modèle selectionné pour 

In [ ]:
best_params_transfer = {
    "lr": trial.params["lr"],
    "optimizer_name": trial.params["optimizer_name"],
    "couches_gel": trial.params["couches_gel"]
}

In [ ]:
print(best_params_transfer)

In [ ]:
val_loss_transfert = []
val_acc_transfert = []
test_acc_transfert = []
test_precision_transfert = []
test_recall_transfert = []

In [ ]:
def transfert():
    mlflow.set_tracking_uri("http://localhost:5000")
    mlflow.set_experiment(params["Experiment"])
    mlflow.config.enable_system_metrics_logging()
    print("✓ Successfully connected to MLflow!")
    with open('requirements.txt', 'r') as file:
        requirements_info = [line.strip() for line in file]
    
    best_val = float("inf")
    with mlflow.start_run(run_name="CNN_transfert_Entrainement_Final"):
        mlflow.log_param("device", device)
        mlflow.log_params(params)
        print("\n********************************\n")
        print(f"Training final model on {device} with best hyperparameters")
        modele = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT).to(device)
        features = list(modele.features.children())

        total_features = len(features)

        layers_to_freeze = min(best_params_transfer["couches_gel"], total_features)

        for i in range(layers_to_freeze):
            for param in features[i].parameters():
                param.requires_grad = False
        
        for i in range(layers_to_freeze, total_features):
            for param in features[i].parameters():
                param.requires_grad = True

        if best_params_transfer["optimizer_name"] == "Adam":
            optimizer = optim.Adam(modele.parameters(), lr=best_params_transfer["lr"])
        if best_params_transfer["optimizer_name"] == "SGD":
            optimizer = optim.SGD(modele.parameters(), lr=best_params_transfer["lr"], momentum=0.9)
        
        scheduler = StepLR(optimizer, step_size=1, gamma=0.7)
        nbre_param = modele.classifier[3].in_features
        modele.classifier[3] = nn.Linear(nbre_param, num_classes)
        modele = modele.to(device)
        
        for epoch in range(1, params["epochs"] + 1):
            tr_loss, tr_acc = train_one_epoch(modele, data_loaders["train"], criterion, optimizer, train_acc)
            val_loss, val_acc = evaluate(modele, data_loaders["val"], criterion, valid_acc)
            print(f"Epoch {epoch:02d} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | val loss {val_loss:.4f} acc {val_acc:.4f}")
            mlflow.log_metrics({"train_loss": tr_loss, "train_accuracy": tr_acc, "val_loss": val_loss, "val_accuracy": val_acc}, step=epoch)
            val_loss_transfert.append(val_loss)
            val_acc_transfert.append(val_acc)
            
            
            
            
            if val_loss < best_val:
                best_val = val_loss
                torch.save(modele.state_dict(), params["save_path_transfert"])
                print("New best model saved!")
            scheduler.step()
        torch.load(params["save_path_transfert"])
        mlflow.pytorch.log_model(modele, name="final_model", pip_requirements="requirements.txt")
    
        # Final test + confusion matrix
        # Final test + confusion matrix
        print("\nEvaluating on test set …")
        cm, test_acc,test_precision, test_recall = test_and_confusion(modele, data_loaders["test"])
        test_acc_transfert.append(test_acc)
        test_precision_transfert.append(test_precision)
        test_recall_transfert.append(test_recall)
        mlflow.log_metric("test_accuracy", test_acc)
        mlflow.log_metric("test_precision", test_precision)
        mlflow.log_metric("test_recall", test_recall)
        plot_confusion_matrix(cm, classes)
        os.replace('confusion_matrix.png', 'confusion_matrix_transfer.png')
        mlflow.log_artifact(os.getcwd()+"\\confusion_matrix_transfer.png", artifact_path="confusion_matrix")
        accuracy_by_class(cm,classes)

        visualize_predictions(modele,data_loaders['test'],classes)
        os.replace('prediction_visualized.png', 'prediction_visualized_transfer.png')
        mlflow.log_artifact(os.getcwd()+"\\prediction_visualized_transfer.png", artifact_path="prediction_visualized")
    return best_val


In [ ]:
best_val_transfert = transfert()

In [ ]:
display(Image(filename='confusion_matrix_transfer.png', width=600))

In [ ]:
display(Image(filename='prediction_visualized_transfer.png', height=600))

### comparaison des deux méthodes

In [ ]:
epoques = list(range(1,21))
plt.figure(figsize=(10, 6))

plt.plot(epoques, val_loss_scratch, color='blue', marker='o', label=' Loss Scratch')
plt.plot(epoques, val_loss_transfert, color='red', marker='s', label='Loss transfert')


plt.title('Comparaison des loss de validation Scratch et Transfer Learning', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Valeur des termes', fontsize=12)
plt.xticks(epoques)
plt.legend()
plt.grid(True)

# 5. Affichage
plt.show()

In [ ]:
epoques = list(range(1,21))
plt.figure(figsize=(10, 6))

plt.plot(epoques, val_acc_scratch, color='blue', marker='o', label=' Accuracy Scratch')
plt.plot(epoques, val_acc_transfert, color='red', marker='s', label='Accuracy transfert')

# 4. Mise en page
plt.title('Comparaison des 2 accuracy de validation Scratch et Transfer Learning', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Valeur des termes', fontsize=12)
plt.xticks(epoques) 
plt.legend()
plt.grid(True)

# 5. Affichage
plt.show()

In [ ]:
tableau_comparatif = pd.DataFrame({
    "Modèle": ["Scratch", "Transfert"],
    "Test Accuracy": [test_acc_scratch[0], test_acc_transfert[0]],
    "Test Precision": [test_precision_scratch[0], test_precision_transfert[0]],
    "Test Recall": [test_recall_scratch[0], test_recall_transfert[0]]
})

In [ ]:


col_labels = tableau_comparatif.columns
cell_text = tableau_comparatif.values

fig, ax = plt.subplots(figsize=(8, 2))

fig.patch.set_visible(False)
ax.axis('off')
ax.axis('tight')

# Création du tableau Matplotlib
table = ax.table(
    cellText=cell_text,
    colLabels=col_labels,
    loc='center',
    cellLoc='center'
)

table.scale(1.2, 2.0)


table.auto_set_font_size(False)


for (row, col), cell in table.get_celld().items():

    cell.set_fontsize(11)
    
    if row == 0:

        cell.set_text_props(weight='bold', color='white')
        cell.set_facecolor('#2C3E50')  
        cell.set_edgecolor('#1A252F')
    else:

        cell.set_text_props(weight='normal', color='#2C3E50')
        cell.set_edgecolor('#BDC3C7')
        

        if row % 2 == 0:
            cell.set_facecolor('#F8F9F9')

plt.show()
